<a href="https://colab.research.google.com/github/BenFeng666/Lab-1/blob/main/testing_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from google.colab import files
import pandas as pd

# 1) Upload your Excel file
uploaded = files.upload()   # select your "Smiles file-Published data.xlsx"
EXCEL_PATH = list(uploaded.keys())[0]  # take uploaded filename

# 2) Read the sheet
df_raw = pd.read_excel(EXCEL_PATH, header=1)

# 3) Keep only the SMILES + efficiency (unchanged!)
df = df_raw[['Structure','transfection efficiency (MC3)']].rename(
    columns={'Structure':'smiles','transfection efficiency (MC3)':'rate'}
).dropna().reset_index(drop=True)

print("Rows after cleaning:", len(df))
print(df.head())


Saving Smiles file-Published data.xlsx to Smiles file-Published data.xlsx
Rows after cleaning: 210
                                              smiles      rate
0  [H][C@]1(OC[C@H]2CCCCN(CC(O)CCCCCCCCCC)CC(O)CC...  2.540036
1  [H][C@]1(OC[C@H]2CCCCN(CCCCCCCC)CCCCCCCC)[C@]2...  0.141439
2  [H][C@]1(OC[C@H]2CCCCN(CCCCCCCCCC)CCCCCCCCCC)[...  2.039738
3  [H][C@]1(OC[C@H]2CCCCN(CCCCCCCCCCCC)CCCCCCCCCC...  0.808456
4  [H][C@]1(OC[C@H]2CCCCN(CCCCCCCCCCCCCC)CCCCCCCC...  0.422100


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import torch

model_name_or_id = "AI4Chem/ChemLLM-7B-Chat"

model = AutoModelForCausalLM.from_pretrained(model_name_or_id, torch_dtype=torch.float16, device_map="auto",trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_id,trust_remote_code=True)

prompt = "What is Molecule of Ibuprofen?"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

generation_config = GenerationConfig(
    do_sample=True,
    top_k=1,
    temperature=0.9,
    max_new_tokens=500,
    repetition_penalty=1.5,
    pad_token_id=tokenizer.eos_token_id
)

outputs = model.generate(**inputs, generation_config=generation_config)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/936 [00:00<?, ?B/s]

configuration_internlm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/AI4Chem/ChemLLM-7B-Chat:
- configuration_internlm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


modeling_internlm2.py: 0.00B [00:00, ?B/s]

...

tokenization_internlm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/AI4Chem/ChemLLM-7B-Chat:
- tokenization_internlm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


./tokenizer.model:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=50, random_state=42, shuffle=True)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Train size:", len(train_df), " Test size:", len(test_df))


train_json = [
    {
        "id": str(i+1),   # +1 so it starts from 1, not 0
        "smiles": r.smiles,
        "transfection_efficiency": float(r.rate)
    }
    for i, r in train_df.iterrows()
]

test_json = [
    {
        "id": str(i+1),
        "smiles": r.smiles,
        "transfection_efficiency": float(r.rate)
    }
    for i, r in test_df.iterrows()
]

print(train_json[:2])  # check



Train size: 160  Test size: 50
[{'id': '1', 'smiles': '[H][C@@]1(OC[C@@H]2CCCCN(CCCCCCCCCCCCCC)CCCCCCCCCCCCCC)[C@@]2([H])OC[C@H]1OCCCN(CCCCCCCCCCCCCC)CCCCCCCCCCCCCC', 'transfection_efficiency': 0.355288614585025}, {'id': '2', 'smiles': 'O=C1C(C(OCCCC)=O)CN(CCCN2CCCC2)CC1C(OCCCC)=O', 'transfection_efficiency': 3.034227}]


In [ ]:
import json
from IPython.display import display, JSON

# This shows a collapsible tree view in Colab
# Print first 30 items from train_json
print("=== Train JSON (first 30) ===")
for item in train_json[:30]:
    print(item)   # each dict on its own line

print("\n=== Test JSON (first 30) ===")
for item in test_json[:50]:
    print(item)



In [ ]:
SYSTEM_PROMPT = """You are a medicinal chemistry and drug delivery assistant.

ontext:
– I will provide a JSON array of training examples from an assay.
– Each item is {{ C"id": "...", "smiles": "...", "transfection_efficiency": <float> }}.
– The goal is to predict the transfection_efficiency (higher=better) for a NEW SMILES under the same assay.

Training examples (paste as-is):
{examples}

Task:
Given the following target SMILES, predict a single numeric transfection_efficiency
and briefly explain the structural rationale (2–5 sentences).

Rules:
– Parse SMILES carefully (including stereochemistry if present).
– Use structure– and motif–level reasoning only (e.g., cationic centers, hydrophobic tails, H-bond donors/acceptors, steric bulk).
– DO NOT output JSON or code. Output plain text: first the number on one line, then 2–5 sentences of explanation.
– If confidence is low due to weak similarity to the training set, say so in the explanation.
""".format(examples=train_json[:5])  # Insert just 5 rows for preview



In [ ]:
def to_chat_row(smiles, rate):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"SMILES: {smiles}"},
            {"role": "assistant", "content": f"{rate:.3f}\nThis molecule has structural features consistent with its activity."}
        ]
    }

train_msgs = [to_chat_row(r.smiles, r.rate) for r in train_df.itertuples()]
test_msgs  = [to_chat_row(r.smiles, r.rate) for r in test_df.itertuples()]

from datasets import Dataset, DatasetDict
ds = DatasetDict({
    "train": Dataset.from_list(train_msgs),
    "test":  Dataset.from_list(test_msgs)
})

print(ds)


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 160
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 50
    })
})


In [ ]:
import json

# use your 150 training rows
train_examples = train_json[:150]

SYSTEM_PROMPT = f"""You are a medicinal chemistry and drug delivery assistant.

Context:
I will provide a JSON array of training examples from an assay.
Each item is {{"id": "...", "smiles": "...", "transfection_efficiency": <float>}}.
The goal is to predict the transfection_efficiency for a NEW SMILES.

Training examples:
{json.dumps(train_examples, indent=2)}

Task:
Given the following target SMILES, predict a single numeric transfection_efficiency
and briefly explain the structural rationale (2–5 sentences).

Rules:
– Output plain text: first the number on one line, then 2–5 sentences of explanation.
– Do NOT output JSON or code.
"""


In [ ]:
import torch, json, re

# How many training examples to include in the prompt (start with 20–40 to avoid OOM)
K = 50                              # change this number as you like
few_examples = train_json[:K]        # uses your 150-row train_json we built earlier


SYSTEM_PROMPT = (
    "You are a medicinal chemistry and drug delivery assistant.\n\n"
    "Context:\n"
    "I will provide a JSON array of training examples...\n"
    "Goal: predict transfection_efficiency for a NEW SMILES.\n\n"
    f"Training examples:\n{json.dumps(few_examples, indent=2)}\n\n"
    "Task:\n"
    "Given the following target SMILES, predict a numeric transfection_efficiency.\n\n"
    "IMPORTANT OUTPUT RULES:\n"
    "1. First line: ONLY the number (float).\n"
    "2. Next 2–5 lines: rationale in full sentences.\n"
    "3. Do NOT write words like 'Prediction:' before the number.\n"
    "4. Do NOT return JSON or code.\n"
)


def build_inputs(smiles: str):
    """Tokenize prompt + query SMILES safely."""
    try:
        # Try using chat template if available
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"SMILES: {smiles}"}
            ],
            add_generation_prompt=True,
            tokenize=False
        )
    except (ValueError, AttributeError):
        # Fallback: plain string if no chat_template
        prompt_text = SYSTEM_PROMPT + f"\n\nSMILES: {smiles}\n"

    enc = tokenizer(prompt_text, return_tensors="pt")
    enc = {k: v.to("cuda") for k, v in enc.items()}

    # Ensure pad token id is set
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    if getattr(model.generation_config, "pad_token_id", None) is None:
        model.generation_config.pad_token_id =_
    return enc



NameError: name 'train_json' is not defined

In [ ]:
import re

# Pick first 10 test examples (change slice as needed)
for i, sample in enumerate(test_json[:2], start=1):
    query_smiles = sample["smiles"]
    true_val = sample["transfection_efficiency"]

    enc = build_inputs(query_smiles)

    # Generate safely
    model.eval()
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=200,
            do_sample=False,
            use_cache=False
        )

    # Decode only new tokens
    input_len = enc["input_ids"].shape[1]
    gen_text = tokenizer.decode(out[0, input_len:], skip_special_tokens=True).strip()

    # Parse number + rationale
    lines = gen_text.splitlines()
    pred_num, rationale = None, ""
    if lines:
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", lines[0])
        if match:
            pred_num = float(match.group(0))
        rationale = "\n".join(lines[1:]).strip()

    # === Print result ===
    print(f"\n=== Sample {i} ===")
    print("SMILES:", query_smiles)
    print(f"Predicted efficiency: {pred_num}")
    print(f"Actual efficiency:    {true_val}")
    print("Rationale:", rationale)



OutOfMemoryError: CUDA out of memory. Tried to allocate 858.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 762.12 MiB is free. Process 823951 has 13.99 GiB memory in use. Of the allocated memory 13.25 GiB is allocated by PyTorch, and 638.91 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import json, random, statistics, torch, re

# bounds from your training data (same units as your sheet)
ymin = float(train_df["rate"].min())
ymax = float(train_df["rate"].max())

# choose the SMILES you want to test
query_smiles = test_df.iloc[0]["smiles"]  # or paste a SMILES string


In [ ]:
def predict_ensemble(smiles, K=30, trials=5, max_new_tokens=200, verbose=False):
    preds, raws = [], []
    for t in range(trials):
        # pick a random subset of training examples for this trial
        subset = random.sample(train_json, k=min(K, len(train_json)))

        # build system prompt with this subset
        sp = (
            "You are a medicinal chemistry and drug delivery assistant.\n\n"
            "Context:\nI will provide a JSON array of training examples from an assay.\n"
            'Each item is {"id": "...", "smiles": "...", "transfection_efficiency": <float>}.\n'
            "The goal is to predict the transfection_efficiency for a NEW SMILES.\n\n"
            f"Training examples:\n{json.dumps(subset, indent=2)}\n\n"
            "Task:\nPredict a single numeric transfection_efficiency on the first line ONLY, "
            "then write 2–5 sentences of explanation.\n\n"
            "Rules:\n– First line must be ONLY the number.\n– Do NOT output JSON or code."
        )

        # tokenize (get real attention_mask)
        msgs = [{"role":"system","content": sp},
                {"role":"user","content": f"SMILES: {smiles}"}]
        prompt_text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        enc = tok(prompt_text, return_tensors="pt")
        enc = {k: v.to("cuda") for k, v in enc.items()}

        # safety: pad token id
        if tok.pad_token_id is None:
            tok.pad_token_id = tok.eos_token_id
        if getattr(model.generation_config, "pad_token_id", None) is None:
            model.generation_config.pad_token_id = tok.pad_token_id

        # generate
        model.eval()
        with torch.inference_mode():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=False)

        # decode only new tokens
        il = enc["input_ids"].shape[1]
        text = tok.decode(out[0, il:], skip_special_tokens=True).strip()
        raws.append(text)

        # parse leading number
        first_line = text.splitlines()[0].strip() if text else ""
        try:
            pred = float(first_line)
            pred = max(ymin, min(ymax, pred))
            pred = float(f"{pred:.3f}")
            preds.append(pred)
        except Exception as e:
            if verbose:
                print(f"[trial {t+1}] parse fail on line: {repr(first_line)} -> {e}")

    if not preds:
        if verbose:
            print("No valid predictions parsed. Here are raw outputs:")
            for i, r in enumerate(raws, 1):
                print(f"\n---- Trial {i} ----\n{r}\n")
        return None, preds, raws

    mean_pred = float(f"{statistics.mean(preds):.3f}")
    return mean_pred, preds, raws


In [ ]:
avg_pred, members, raws = predict_ensemble(query_smiles, K=30, trials=5, verbose=True)

print("\n=== ENSEMBLE RESULT ===")
print("SMILES:", query_smiles[:120] + ("..." if len(query_smiles) > 120 else ""))
print("Mean prediction:", avg_pred)
print("Members:", members)

# Optional: show the first raw output (number + rationale) to inspect
if raws:
    print("\n--- Raw output (trial 1) ---")
    print(raws[0])


[trial 1] parse fail on line: 'Predicted transfection_efficiency: 2.2' -> could not convert string to float: 'Predicted transfection_efficiency: 2.2'


KeyboardInterrupt: 